In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# CSV 文件名
filename = 'BAT_Heat_Log_Data_2026_06_22_16_19_45.csv'

# 读取 CSV
df = pd.read_csv(filename, encoding='utf-8')
df.columns = df.columns.str.strip()  # 去掉列名两端空格

# 检查列名
required = ['Accel_x_h', 'Accel_y_h', 'Accel_z_h']
for col in required:
    if col not in df.columns:
        raise ValueError(f'CSV 中找不到列: {col}')



# 绘图
plt.figure(figsize=(10, 6))
plt.plot(df.index, df['Accel_x_h'], label='x')
plt.plot(df.index, df['Accel_y_h'], label='y')
plt.plot(df.index, df['Accel_z_h'], label='z')
plt.xlabel('样本点')
plt.ylabel('加速度')
plt.title('x / y / z 可视化')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# CSV 文件名
filename = 'BAT_Heat_Log_Data_2026_06_22_16_19_45.csv'

# 读取 CSV
df = pd.read_csv(filename, encoding='utf-8')
df.columns = df.columns.str.strip()  # 去掉列名两端空格

# 检查列名
required = ['gyro_x', 'gyro_y', 'gyro_z']
for col in required:
    if col not in df.columns:
        raise ValueError(f'CSV 中找不到列: {col}')



# 绘图
plt.figure(figsize=(10, 6))
plt.plot(df.index, df['Accel_x'], label='x')
plt.plot(df.index, df['Accel_y'], label='y')
plt.plot(df.index, df['Accel_z'], label='z')
plt.xlabel('样本点')
plt.ylabel('加速度')
plt.title('x / y / z 可视化')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ================== 配置区（你只需要改这里） ==================
file_name = "BAT_Heat_Log_Data_2026_06_15_14_17_43_imu_result.csv"       # CSV 文件名
columns_to_read = ["lax", "lay", "laz"]  # 要读取的三列列名

# 采样间隔（单位：ms）
time_interval_ms = 20

# ================== 读取数据 ==================
df = pd.read_csv(file_name)

# 校验列是否存在
for col in columns_to_read:
    if col not in df.columns:
        raise ValueError(f"列 {col} 不在CSV中！")

data = df[columns_to_read]

# ================== 构建 X 轴 ==================
num_points = len(data)
x = np.arange(10, num_points * time_interval_ms, time_interval_ms)

# ================== 自动计算 Y 轴范围 ==================
y_min = data.min().min()/ 1000 * 9.8
y_max = data.max().max()/ 1000 * 9.8

# 稍微留一点边界
margin = (y_max - y_min) * 0.05
y_min -= margin
y_max += margin

# ================== 绘图 ==================
plt.figure(figsize=(10, 5))

for col in columns_to_read:
    plt.plot(x, data[col]/1000 *9.8, label=col)

plt.xlabel("Time (ms)")
plt.ylabel("Value")
plt.title("raw data near 10m/s2")

plt.ylim(y_min, y_max)

plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ================== 配置区（你只需要改这里） ==================
file_name = "BAT_Heat_Log_Data_2026_06_17_18_44_07_imu_result.csv"       # CSV 文件名
columns_to_read = ["q1", "q2", "q3"]  # 要读取的三列列名

# 采样间隔（单位：ms）
time_interval_ms = 20

# ================== 读取数据 ==================
df = pd.read_csv(file_name)

# 校验列是否存在
for col in columns_to_read:
    if col not in df.columns:
        raise ValueError(f"列 {col} 不在CSV中！")

data = df[columns_to_read]

# ================== 构建 X 轴 ==================
num_points = len(data)
x = np.arange(1, num_points * time_interval_ms, time_interval_ms)

# ================== 自动计算 Y 轴范围 ==================
y_min = data.min().min()
y_max = data.max().max()

# 稍微留一点边界
margin = (y_max - y_min) * 0.05
y_min -= margin
y_max += margin

# ================== 绘图 ==================
plt.figure(figsize=(10, 5))

for col in columns_to_read:
    plt.plot(x, data[col], label=col)

plt.xlabel("Time (ms)")
plt.ylabel("Value")
plt.title("linear acc near 1m/s2")

plt.ylim(y_min, y_max)

plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ================== 配置区（你只需要改这里） ==================
file_name = "low_allmove_imu_result_26.6.11.csv"       # CSV 文件名
columns_to_read = ["lax", "lay", "laz"]               # 要读取的三列列名

# 采样间隔（单位：ms）
time_interval_ms = 20

# ================== 高通滤波器参数 ==================
# 50 Hz采样、1 Hz截止的一阶Butterworth高通
b0 = 0.940809
b1 = -0.940809
a1 = -0.881619

def highpass_filter(signal, b0, b1, a1):
    signal = np.asarray(signal, dtype=float)
    y = np.zeros_like(signal)

    # 初始化：避免开头全0太明显，可直接让首点跟输入一致
    if len(signal) > 0:
        y[0] = signal[0]

    for n in range(1, len(signal)):
        y[n] = b0 * signal[n] + b1 * signal[n - 1] - a1 * y[n - 1]

    return y

# ================== 读取数据 ==================
df = pd.read_csv(file_name)

# 校验列是否存在
for col in columns_to_read:
    if col not in df.columns:
        raise ValueError(f"列 {col} 不在CSV中！")

data = df[columns_to_read].copy()

# ================== 对每一列做高通滤波 ==================
filtered_data = pd.DataFrame()
for col in columns_to_read:
    filtered_data[col] = highpass_filter(data[col].values, b0, b1, a1)

# ================== 构建 X 轴 ==================
num_points = len(filtered_data)
x = np.arange(0, num_points * time_interval_ms, time_interval_ms)

# ================== 自动计算 Y 轴范围 ==================
y_min = filtered_data.min().min()
y_max = filtered_data.max().max()

margin = (y_max - y_min) * 0.05
if margin == 0:
    margin = 1e-6
y_min -= margin
y_max += margin

# ================== 绘图 ==================
plt.figure(figsize=(10, 5))

for col in columns_to_read:
    plt.plot(x, filtered_data[col], label=f"{col}_hp")

plt.xlabel("Time (ms)")
plt.ylabel("Value")
plt.title("Linear Acceleration with High-Pass Filter (50Hz, 1Hz cutoff)")
plt.ylim(y_min, y_max)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
